# Mission 3: (AI edge case testing)

1. **Does the new-task data load correctly?** Each entry should be a full story with its header lines, not a single line.
2. **Are stories cut off?** Stories longer than `common.MAX_LENGTH` tokens are truncated in training and evaluation.
3. **Which `m` to use?** For each candidate `m`, it runs phase 1 (plain stories) and then phase 2 (new task) with plain fine-tuning, and measures how much the model forgets.
4. **What bar should "improve on the new task" use?** It reports the new-task perplexity that plain fine-tuning reaches.

Run on a Colab T4 GPU with `common.py` uploaded. Expected runtime: a few minutes per value of `m`. Do not release this notebook to students.

## 1. Setup

In [1]:
!pip install -q transformers datasets pandas

import os, time
import pandas as pd
assert os.path.exists("common.py"), "Put common.py in the same folder as this notebook."
import common

common.set_seed()
print("Device:", common.get_device())

Device: cuda


## 2. Load both tasks

In [2]:
original, val_original = common.load_tinystories()
new, val_new = common.load_new_task()

Loaded 10,000 training stories and 1,000 validation stories.
Loaded 5,000 new-task training entries and 500 validation entries.


## 3. Check the new-task format

Each entry should contain several header lines and a `Story:` line. If entries are single short lines (for example, just `Words: oak, quit, gloomy`), the dataset stores one line per row, and `load_new_task()` needs to regroup rows into stories before release.

In [3]:
for i in range(3):
    print(f"--- new-task entry {i} ---")
    print(new[i][:500])
    print()

has_story = sum("Story:" in e for e in new) / len(new)
lines_per_entry = sum(e.count("\n") + 1 for e in new) / len(new)
print(f"Entries containing 'Story:': {has_story:.0%}")
print(f"Average lines per entry:     {lines_per_entry:.1f}")
if has_story < 0.9 or lines_per_entry < 3:
    print("WARNING: entries look like single lines, not full stories. Fix load_new_task() before release.")
else:
    print("OK: entries look like full stories with headers.")

--- new-task entry 0 ---
Features: Dialogue
Words: quit, oak, gloomy
Summary: Sara and Ben were playing in the park, but Sara wanted to go home because it was cold and dark. Ben convinced her to stay and play, but eventually agreed to go home and have hot cocoa.
Story:
Sara and Ben were playing in the park. They liked to climb the big oak tree and pretend they were birds. They made nests with leaves and twigs and sang songs.
But today, the sky was gloomy and the wind was cold. Sara felt sad and cold. She wanted to go ho

--- new-task entry 1 ---
Summary: Lily steals a new bike from a store and gets into an accident while riding it, resulting in her getting hurt and being sent to jail, losing her parents' trust and love. The moral of the story is not to steal.
Words: ride, work, upset
Features: Dialogue, BadEnding, MoralValue
Story:
Lily liked to ride her bike. She rode it every day after work. Work was a place where she helped her mom and dad with chores. She liked work, but she liked 

## 4. Check truncation

Training and evaluation use only the first `common.MAX_LENGTH` tokens of each story. If many new-task entries are longer, consider raising `MAX_LENGTH` in `common.py`.

In [4]:
model, tokenizer = common.load_model()

def share_truncated(texts):
    lengths = [len(tokenizer(t)["input_ids"]) for t in texts]
    return sum(n > common.MAX_LENGTH for n in lengths) / len(lengths), sorted(lengths)[len(lengths) // 2]

for name, texts in [("Original task", val_original), ("New task", val_new)]:
    share, median = share_truncated(texts)
    print(f"{name:<14} median length {median:4d} tokens | longer than {common.MAX_LENGTH}: {share:.0%}")

Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

Original task  median length  180 tokens | longer than 384: 3%
New task       median length  236 tokens | longer than 384: 2%


## 5. Settings

`M_VALUES` are the phase-2 lengths to try. `FORGETTING_THRESHOLD` is how much original-task perplexity must rise (relative to after phase 1) to count as clear forgetting. 10% is a rule of thumb, not a standard; adjust it if needed.

In [5]:
M_VALUES = [100, 200, 500]
FORGETTING_THRESHOLD = 0.10
N_STEPS = common.M3_N_STEPS

## 6. Phase 1 (once)

Train the base model on plain stories for `N_STEPS` steps. Every value of `m` starts from this same phase-1 model, so the comparison is fair.

In [6]:
def evaluate(m):
    return (common.compute_perplexity(m, tokenizer, val_original),
            common.compute_perplexity(m, tokenizer, val_new))

base_orig, base_new = evaluate(model)
print(f"Base model:    original-task ppl {base_orig:.2f} | new-task ppl {base_new:.2f}")

common.finetune(model, tokenizer, original, epochs=100, max_steps=N_STEPS)
# common.finetune(model, tokenizer, original, epochs=100, max_steps=N_STEPS, lr=1e-5)
p1_orig, p1_new = evaluate(model)
print(f"After phase 1: original-task ppl {p1_orig:.2f} | new-task ppl {p1_new:.2f}")

# Keep a copy of the phase-1 weights so each m starts from the same point.
phase1_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

Base model:    original-task ppl 3.53 | new-task ppl 12.00
  step    50 | epoch 1 | loss 1.1768
  step   100 | epoch 1 | loss 1.1286
  step   150 | epoch 1 | loss 1.1569
  step   200 | epoch 1 | loss 1.1163
Finished: 200 steps in 74.2s, final loss 1.2314
After phase 1: original-task ppl 3.43 | new-task ppl 11.16


## 7. Phase 2 for each value of `m`

In [7]:
rows = []
for m in M_VALUES:
    m_model, _ = common.load_model()
    m_model.load_state_dict(phase1_state)
    start = time.time()
    common.finetune(m_model, tokenizer, new, epochs=100, max_steps=m, log_every=0)
    seconds = time.time() - start
    orig, new_ppl = evaluate(m_model)
    rows.append({
        "m": m,
        "original ppl": round(orig, 2),
        "forgetting (vs phase 1)": f"{orig / p1_orig - 1:+.0%}",
        "new-task ppl": round(new_ppl, 2),
        "new-task change (vs base)": f"{new_ppl / base_new - 1:+.0%}",
        "phase-2 time (s)": round(seconds),
        "_forgetting": orig / p1_orig - 1,
        "_new_improved": new_ppl < base_new,
    })
    del m_model

results = pd.DataFrame(rows)
results.drop(columns=["_forgetting", "_new_improved"])

Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

Finished: 100 steps in 37.4s, final loss 1.4445


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

Finished: 200 steps in 75.2s, final loss 1.4884


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

Finished: 500 steps in 187.5s, final loss 1.3444


,m,original ppl,forgetting (vs phase 1),new-task ppl,new-task change (vs base),phase-2 time (s)
0,100,3.62,+6%,4.53,-62%,37
1,200,3.73,+9%,4.23,-65%,75
2,500,3.93,+15%,3.83,-68%,188


## 8. Recommendation

In [8]:
ok = results[(results["_forgetting"] >= FORGETTING_THRESHOLD) & results["_new_improved"]]
if len(ok):
    best = ok.iloc[0]
    print(f"Recommended M3_M_STEPS = {int(best['m'])}")
    print(f"  Plain fine-tuning raises original-task perplexity by {best['forgetting (vs phase 1)']} (threshold {FORGETTING_THRESHOLD:.0%}),")
    print(f"  and reaches new-task perplexity {best['new-task ppl']} (base model: {base_new:.2f}).")
    print("  Set M3_M_STEPS in common.py to this value.")
else:
    most = results.loc[results["_forgetting"].idxmax()]
    print(f"No value of m reached {FORGETTING_THRESHOLD:.0%} forgetting with new-task improvement.")
    print(f"  Largest forgetting: {most['forgetting (vs phase 1)']} at m={int(most['m'])}.")
    print("  Try larger values in M_VALUES, a higher learning rate in phase 2, or a lower threshold.")

Recommended M3_M_STEPS = 500
  Plain fine-tuning raises original-task perplexity by +15% (threshold 10%),
  and reaches new-task perplexity 3.83 (base model: 12.00).
  Set M3_M_STEPS in common.py to this value.


## 9. Values update
in common.py
- learning rate 1e-4 to 1e-5
- MAX_LENGTH from 256 to 384
- Test M_steps for 100, 200, 500 and final value set to 500